In [ ]:
Extraer información de RemoteOK, usnado API publica 

In [ ]:
# ----------------------------
# 0️⃣ Librerías
# ----------------------------
import requests
import pandas as pd
import os
import re
from datetime import datetime

# ----------------------------
# 1️⃣ Carpeta para guardar CSV
# ----------------------------
BASE_DIR = os.path.dirname(os.path.abspath(""))  # carpeta del Notebook
RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
os.makedirs(RAW_DATA_DIR, exist_ok=True)

# ----------------------------
# 2️⃣ URL API RemoteOK
# ----------------------------
API_URL = "https://remoteok.com/api"

# ----------------------------
# 3️⃣ Niveles y patrones de experiencia
# ----------------------------
xp_levels = ["junior", "mid-level", "mid level", "senior", "lead", "manager"]

# Patrones de inglés
english_levels = [
    "english", "fluent english", "english required",
    "intermediate english", "advanced english"
]

# ----------------------------
# 4️⃣ Funciones auxiliares
# ----------------------------
def detect_years(text):
    """Detecta años de experiencia en el texto."""
    pattern = r"(\d+)[\+\-]?\d*\s+years?"
    match = re.search(pattern, text.lower())
    if match:
        return match.group(0)
    return ""

def detect_english(text):
    """Detecta nivel de inglés en el texto."""
    for level in english_levels:
        if level.lower() in text.lower():
            return level
    return ""

# ----------------------------
# 5️⃣ Descargar datos de RemoteOK
# ----------------------------
response = requests.get(API_URL, headers={"User-Agent": "Mozilla/5.0"})
if response.status_code != 200:
    raise Exception(f"Error al conectar con RemoteOK: {response.status_code}")

data = response.json()
jobs = data[1:]  # ignorar primera fila general

# ----------------------------
# 6️⃣ Procesar todas las vacantes
# ----------------------------
all_jobs = []

for job in jobs:
    combined_text = " ".join([
        str(job.get("position", "")),
        " ".join(job.get("tags", [])),
        str(job.get("company", "")),
        str(job.get("description", "")) if "description" in job else ""
    ])
    
    # Nivel de experiencia
    level = ""
    for xp in xp_levels:
        if xp in combined_text.lower():
            level = xp
            break
    
    # Años de experiencia
    years = detect_years(combined_text)
    
    # Nivel de inglés
    english = detect_english(combined_text)
    
    all_jobs.append({
        "date": job.get("date"),
        "company": job.get("company"),
        "position": job.get("position"),
        "location": job.get("location"),
        "tags": ", ".join(job.get("tags", [])),
        "remote": job.get("remote"),
        "experience_level": level,
        "experience_years": years,
        "english_level": english,
        "url": job.get("url")
    })

# ----------------------------
# 7️⃣ Guardar CSV en data/raw/
# ----------------------------
df = pd.DataFrame(all_jobs)
today = datetime.today().strftime("%Y_%m_%d")
csv_file = os.path.join(RAW_DATA_DIR, f"remoteok_jobs_all_{today}.csv")
df.to_csv(csv_file, index=False)

print(f"✅ {len(df)} vacantes guardadas en {csv_file}")


_IncompleteInputError: incomplete input (1794761727.py, line 106)